In [ ]:
# ============================================================
# train_refinedSOS.ipynb
#
# DeepLabV3-ResNet50 training script for SAR oil spill segmentation
# Dataset: Refined SOS
# Dataset structure:
# data_root/
# ├── images/
# │   ├── train/
# │   └── val/
# └── masks/
#     ├── train/
#     └── val/


# Stage 1 training script.

# Train DeepLabV3 using the Refined SOS dataset.

# Input:
#     Refined SOS images and masks

# Output:
#    deeplabv3_refinedSOS.pth

# ============================================================

In [ ]:
!pip3 install http://download.pytorch.org/whl/cu80/torch-0.3.0.post4-cp36-cp36m-linux_x86_64.whl
!pip3 install torchvision

ERROR: torch-0.3.0.post4-cp36-cp36m-linux_x86_64.whl is not a supported wheel on this platform.


In [ ]:
# 1. Google Drive Mount
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models.segmentation import deeplabv3_resnet50

from tqdm import tqdm

In [ ]:
# ============================================================
# 3. Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
# ============================================================
# Copy refined SOS to local SSD
# ============================================================

import shutil
import os

localRoot = "/content/refined SOS"

if not os.path.exists(localRoot):

    print("Copying refined SOS to local SSD...")

    shutil.copytree(
        "/content/drive/MyDrive/SAR_AI_Oil_Spill_Detection/refined SOS",
        localRoot
    )

    print("Copy complete.")

else:
    print("Local dataset already exists.")

Copying refined SOS to local SSD...
Copy complete.


In [ ]:
# ============================================================
# 4. Dataset Path
# ============================================================

# ============================================================
# Use local dataset instead of Drive
# ============================================================

rootDir = "/content/refined SOS"

trainImageDir = os.path.join(rootDir, "images", "train")
trainMaskDir  = os.path.join(rootDir, "masks",  "train")

valImageDir = os.path.join(rootDir, "images", "val")
valMaskDir  = os.path.join(rootDir, "masks",  "val")

In [ ]:
# ============================================================
# 5. Dataset Class
# ============================================================

class OilSpillDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = image_dir
        self.mask_dir = mask_dir

        self.image_files = sorted([
            f for f in os.listdir(image_dir)
            if f.endswith(".png")
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):

        img_name = self.image_files[idx]

        img_path = os.path.join(self.image_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        # -----------------------------
        # Read image
        # -----------------------------
        image = Image.open(img_path)

        image = np.array(image)

        # grayscale → RGB
        if len(image.shape) == 2:
            image = np.stack([image]*3, axis=-1)

        image = image.astype(np.float32)

        # normalize
        image = image / 255.0

        # HWC -> CHW
        image = torch.tensor(image).permute(2,0,1)

        # -----------------------------
        # Read mask
        # -----------------------------
        mask = Image.open(mask_path)

        mask = np.array(mask)

        # RGB mask -> single channel
        if len(mask.shape) == 3:
            mask = mask[:,:,0]

        # binary mask
        mask = (mask > 0).astype(np.uint8)

        mask = torch.tensor(mask).long()

        return image, mask

In [ ]:
# ============================================================
# 6. Dataset / DataLoader
# ============================================================

train_dataset = OilSpillDataset(trainImageDir, trainMaskDir)
val_dataset   = OilSpillDataset(valImageDir, valMaskDir)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 6455
Val: 1615


In [ ]:
# ============================================================
# 7. DeepLabV3 Model
# ============================================================

model = deeplabv3_resnet50(
    weights=None,
    num_classes=2
)

model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 182MB/s]


In [ ]:
# 8. Loss Function
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
# ============================================================
# 9. Training
# ============================================================

num_epochs = 10

train_losses = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    loop = tqdm(train_loader)

    for images, masks in loop:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)['out']

        loss = criterion(outputs, masks)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        loop.set_description(f"Epoch [{epoch+1}/{num_epochs}]")
        loop.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

    # ============================================================
# 10. Save Model
# ============================================================

save_path = "/content/drive/MyDrive/SAR_AI_Oil_Spill_Detection/DeeplabV3_baseline.pth"

torch.save(model.state_dict(), save_path)

print("Model saved:", save_path)


# ============================================================
# 11. Inference Test
# ============================================================

model.eval()

test_img, test_mask = val_dataset[0]

with torch.no_grad():

    input_tensor = test_img.unsqueeze(0).to(device)

    output = model(input_tensor)['out']

    pred = torch.argmax(output, dim=1)

pred = pred.squeeze().cpu().numpy()

test_img_np = test_img.permute(1,2,0).numpy()

test_mask_np = test_mask.numpy()


Epoch [1/10]: 100%|██████████| 404/404 [10:01<00:00,  1.49s/it, loss=0.138]


Epoch 1 Loss: 0.1049


Epoch [2/10]: 100%|██████████| 404/404 [10:00<00:00,  1.49s/it, loss=0.117]


Epoch 2 Loss: 0.1221


Epoch [3/10]: 100%|██████████| 404/404 [10:00<00:00,  1.49s/it, loss=0.083]


...

Epoch 8 Loss: 0.0735


Epoch [9/10]: 100%|██████████| 404/404 [10:01<00:00,  1.49s/it, loss=0.0861]


Epoch 9 Loss: 0.0699


Epoch [10/10]: 100%|██████████| 404/404 [10:01<00:00,  1.49s/it, loss=0.0721]


Epoch 10 Loss: 0.0675
Model saved: /content/drive/MyDrive/SAR_AI_Oil_Spill_Detection/DeeplabV3_baseline.pth
